# Install & imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


# Reconnect paths, build test set (features + adjacency)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PD_EEG_Thesis_Features_v2')
FEAT_DIR     = PROJECT_ROOT / 'data' / 'features'
SEG_DIR      = PROJECT_ROOT / 'data' / 'segments'
SPLIT_DIR    = PROJECT_ROOT / 'data' / 'split'

d_full = np.load(str(FEAT_DIR / 'features_all.npz'))
node_features_full, y_full, subjects_full = d_full['node_features'], d_full['y'], d_full['subjects']

seg_d = np.load(str(SEG_DIR / 'segments_all.npz'))
X_raw, subjects_raw = seg_d['X'], seg_d['subjects']

split_df  = pd.read_csv(SPLIT_DIR / 'subject_split.csv')
test_subj = split_df.loc[split_df['split']=='test','subject'].astype(str).values

test_mask     = np.isin(subjects_full, test_subj)
test_mask_raw = np.isin(subjects_raw, test_subj)

nf_test, y_test, subj_test = node_features_full[test_mask], y_full[test_mask], subjects_full[test_mask]

print(f'Test set: {nf_test.shape}   PD: {int(y_test.sum())}   Control: {int((y_test==0).sum())}')
print('Note: no balancing operation applied — class weighting (chosen method) has no test-time')
print('equivalent. The test set remains exactly as segmented, real and untouched.')

Mounted at /content/drive
Test set: (627, 6, 60, 17)   PD: 417   Control: 210
Note: no balancing operation applied — class weighting (chosen method) has no test-time
equivalent. The test set remains exactly as segmented, real and untouched.


# Compute adjacency for test windows

In [3]:
CORR_THRESHOLD = 0.3

def compute_adjacency(window):
    corr = np.nan_to_num(np.corrcoef(window), nan=0.0)
    adj  = np.where(corr >= CORR_THRESHOLD, corr, 0.0).astype(np.float32)
    np.fill_diagonal(adj, 0.0)
    return adj

adj_test = np.stack([compute_adjacency(X_raw[i]) for i in np.where(test_mask_raw)[0]])
print(f'Test adjacency: {adj_test.shape}')
assert adj_test.shape[0] == nf_test.shape[0]

Test adjacency: (627, 60, 60)


# Save

In [4]:
np.savez_compressed(str(SPLIT_DIR / 'test_data.npz'),
                    node_features=nf_test, adj=adj_test, y=y_test, subjects=subj_test)
print(f'Saved test_data.npz: {nf_test.shape}')

Saved test_data.npz: (627, 6, 60, 17)
